# Boosted Decision Tree

GBDT costruisce alberi in modo sequenziale, correggendo progressivamente gli errori di classificazione del modello costruito fino a quel punto, producendo un modello complesso e molto accurato

In [4]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from pathlib import Path
import warnings
# Nascondo i warning
warnings.filterwarnings('ignore')
# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Treining


In [5]:
def training(file_path, csv_name):
    # Leggo i csv
    df = pd.read_csv(file_path)

    # Filtro solo le pazienti con PR valido
    df_PRvalido = df[df['PR [SII]'].notna()].copy()

    # Vado a separare le features e target
    features = df_PRvalido.drop(columns=['Patient ID', 'lesion idx', 'tumor/benign', 
                             'GRADE', 'ER [SII]', 'PR [SII]', 'HER2 [SII]', 
                             'isTN', 'KI67 [%]', 'Breast'])

    # Prendo solo PR [SII], convertita in valori interi.
    target = df_PRvalido['PR [SII]'].astype(int)

    # Normalizzo i dati per evitare problemi di scala
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # Istanzio il modello Gradient Boosting Decision Tree
    gboost = GradientBoostingClassifier(
        n_estimators=100,           # Numero di alberi
        learning_rate=0.1,          # Tasso di apprendimento
        max_depth=3,                # Profondità massima degli alberi
        min_samples_split=5,        # Minimo campioni per split
        min_samples_leaf=2,         # Minimo campioni per foglia
        subsample=0.8,              # Frazione campioni per albero
        random_state=42
    )

    # Alleno il modello sui dati normalizzati
    gboost.fit(features_scaled, target)

    # Creo la cross-validation a 5 fold stratificata
    cv = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
    scores = cross_val_score(gboost, features_scaled, target, cv=cv, scoring='accuracy')

    # Ritorno risultati sintetici: media, deviazione standard e score per ogni fold
    return {
        'mean_accuracy': scores.mean(),     # Accuratezza media su tutte le fold
        'std_accuracy': scores.std(),       # Variabilità tra le fold
        'scores_per_fold': scores           # Accuratezza tra ciascun fold
    }

# Lettura dei file

In [6]:
print("="*50 +"\nTRAINING GRADIENT BOOSTING DECISION TREE\n" + "="*50)
results = {}
for name, file_path in datasets.items():
    print(f"\n{name}")
    results[name] = training(file_path, name)
    print(f"Accuracy media: {results[name]['mean_accuracy']:.3f} ± {results[name]['std_accuracy']:.3f}")
    print(f"Scores per fold: {[f'{s:.3f}' for s in results[name]['scores_per_fold']]}")

TRAINING GRADIENT BOOSTING DECISION TREE

t2_medsam
Accuracy media: 0.390 ± 0.145
Scores per fold: ['0.200', '0.400', '0.200', '0.400', '0.600', '0.200', '0.600', '0.500', '0.250', '0.250', '0.500', '0.500', '0.500', '0.500', '0.250']

t2_preprocessed
Accuracy media: 0.403 ± 0.219
Scores per fold: ['0.000', '0.400', '0.200', '0.400', '0.400', '0.200', '0.200', '0.500', '0.750', '0.250', '0.500', '0.750', '0.750', '0.500', '0.250']

t2_original
Accuracy media: 0.420 ± 0.184
Scores per fold: ['0.200', '0.600', '0.400', '0.600', '0.400', '0.200', '0.400', '0.500', '0.750', '0.000', '0.500', '0.500', '0.500', '0.500', '0.250']

medsam_dynamic
Accuracy media: 0.450 ± 0.263
Scores per fold: ['0.400', '0.400', '0.200', '0.200', '0.200', '0.400', '0.200', '0.750', '0.750', '0.500', '1.000', '0.500', '0.750', '0.500', '0.000']

preprocessed_dynamic
Accuracy media: 0.400 ± 0.208
Scores per fold: ['0.000', '0.600', '0.200', '0.200', '0.200', '0.400', '0.400', '0.750', '0.750', '0.250', '0.500', '